In [71]:
from pathlib import Path
import os

# Detect project root robustly
PROJECT_ROOT = Path.cwd()

# If running from frontend/, go one level up
if PROJECT_ROOT.name == "frontend":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ART_DIR  = PROJECT_ROOT / "artifacts"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
ART_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Artifacts dir:", ART_DIR)

Project root: /Users/adarshthakur/Downloads/MIANALYZER
Data dir: /Users/adarshthakur/Downloads/MIANALYZER/data
Artifacts dir: /Users/adarshthakur/Downloads/MIANALYZER/artifacts


In [72]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

DATA_PATH = DATA_DIR / "hn_market_relevant_latest.csv"

OUT_VECTORIZER = ART_DIR / "tfidf_vectorizer.joblib"
OUT_KMEANS     = ART_DIR / "kmeans_model.joblib"
OUT_THEME_MAP  = ART_DIR / "cluster_name_map.json"

K = 8
RANDOM_STATE = 42

print("Training input:", DATA_PATH)
print("Vectorizer will save to:", OUT_VECTORIZER)
print("KMeans will save to:", OUT_KMEANS)
print("Theme map will save to:", OUT_THEME_MAP)

Training input: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_market_relevant_latest.csv
Vectorizer will save to: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/tfidf_vectorizer.joblib
KMeans will save to: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/kmeans_model.joblib
Theme map will save to: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/cluster_name_map.json


In [73]:
df = pd.read_csv(DATA_PATH)

required_cols = ["clean_text"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
df["clean_text"] = df["clean_text"].astype(str).fillna("")
df = df[df["clean_text"].str.len() >= 40].copy()  # consistent with your cleaning logic
print("Training rows:", len(df))

Training rows: 874


In [74]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=6000,
    ngram_range=(1, 2),
    min_df=3
)

X = vectorizer.fit_transform(df["clean_text"].tolist())
print("TFIDF shape:", X.shape)

TFIDF shape: (874, 5046)


In [75]:
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init="auto")
clusters = kmeans.fit_predict(X)
df["cluster"] = clusters
print(df["cluster"].value_counts().sort_index())

cluster
0    203
1    381
2     18
3     21
4     20
5     46
6    137
7     48
Name: count, dtype: int64


In [76]:
CLUSTER_NAME_MAP = {
    0: "Pricing & Monetization Concerns",
    1: "Early Product Feedback & Tool Discovery",
    2: "Platform Perception & Social Sentiment",
    3: "Ad-Based Monetization Trade-offs",
    4: "Developer Ecosystem & Integrations",
    5: "AI Content Platforms & Automation",
    6: "Creator Economy & AI Disruption",
    7: "Open Platform & Trust Signals"
}

# safety check
if set(CLUSTER_NAME_MAP.keys()) != set(range(K)):
    raise ValueError("CLUSTER_NAME_MAP keys must be 0..K-1")

In [77]:
joblib.dump(vectorizer, OUT_VECTORIZER)
joblib.dump(kmeans, OUT_KMEANS)

with open(OUT_THEME_MAP, "w") as f:
    json.dump(CLUSTER_NAME_MAP, f, indent=2)

print("Saved:", OUT_VECTORIZER, OUT_KMEANS, OUT_THEME_MAP)

Saved: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/tfidf_vectorizer.joblib /Users/adarshthakur/Downloads/MIANALYZER/artifacts/kmeans_model.joblib /Users/adarshthakur/Downloads/MIANALYZER/artifacts/cluster_name_map.json


<h1> Daily Insights</h1>

In [78]:
import json
import re
import joblib
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Paths (automation-safe) ---
IN_PATH = DATA_DIR / "hn_market_relevant_latest.csv"

OUT_SCORED   = DATA_DIR / "hn_scored_latest.csv"
OUT_INSIGHTS = DATA_DIR / "insights_latest.csv"
OUT_QUOTES   = DATA_DIR / "quotes_latest.csv"

VECTORIZER_PATH = ART_DIR / "tfidf_vectorizer.joblib"
KMEANS_PATH     = ART_DIR / "kmeans_model.joblib"
THEME_MAP_PATH  = ART_DIR / "cluster_name_map.json"

print("IN_PATH:", IN_PATH)
print("OUT_SCORED:", OUT_SCORED)
print("VECTORIZER_PATH:", VECTORIZER_PATH)
print("KMEANS_PATH:", KMEANS_PATH)
print("THEME_MAP_PATH:", THEME_MAP_PATH)

IN_PATH: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_market_relevant_latest.csv
OUT_SCORED: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_scored_latest.csv
VECTORIZER_PATH: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/tfidf_vectorizer.joblib
KMEANS_PATH: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/kmeans_model.joblib
THEME_MAP_PATH: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/cluster_name_map.json


In [79]:
df = pd.read_csv(IN_PATH)
df["clean_text"] = df["clean_text"].astype(str).fillna("")

vectorizer = joblib.load(VECTORIZER_PATH)
kmeans = joblib.load(KMEANS_PATH)

with open(THEME_MAP_PATH) as f:
    theme_map = json.load(f)

# json keys are strings -> convert to int keys
theme_map = {int(k): v for k, v in theme_map.items()}

In [80]:
X = vectorizer.transform(df["clean_text"].tolist())
df["cluster"] = kmeans.predict(X)
df["market_theme"] = df["cluster"].map(theme_map)

In [81]:
def add_hn_item_url(df, objectid_col="objectID", url_col="url"):
    df = df.copy()
    if objectid_col in df.columns:
        hn_link = "https://news.ycombinator.com/item?id=" + df[objectid_col].astype(str)
        if url_col not in df.columns:
            df[url_col] = hn_link
        else:
            df[url_col] = df[url_col].fillna(hn_link)
    return df

df = add_hn_item_url(df)

In [82]:
def representative_quotes(df, theme_col="market_theme", text_col="clean_text", n_quotes=3, min_len=60):
    results = []
    for theme, g in df.dropna(subset=[theme_col, text_col]).groupby(theme_col):
        g = g[g[text_col].astype(str).str.len() >= min_len].copy()
        if g.empty:
            continue

        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=6000)
        X = vec.fit_transform(g[text_col].astype(str).tolist())
        centroid = np.asarray(X.mean(axis=0))
        sims = cosine_similarity(X, centroid).ravel()

        top_idx = np.argsort(sims)[::-1][:n_quotes]
        for rank, idx in enumerate(top_idx, start=1):
            row = g.iloc[idx]
            results.append({
                "market_theme": theme,
                "rank": rank,
                "keyword": row.get("keyword", ""),
                "story_title": row.get("story_title", ""),
                "author": row.get("author", ""),
                "url": row.get("url", ""),
                "objectID": row.get("objectID", ""),
                "clean_text": row.get(text_col, "")
            })
    return pd.DataFrame(results).sort_values(["market_theme", "rank"])

quotes_df = representative_quotes(df, n_quotes=3)

In [83]:
def theme_stats(df):
    tmp = df.copy()
    tmp["created_at"] = pd.to_datetime(tmp.get("created_at", None), errors="coerce")
    stats = (
        tmp.groupby("market_theme")
        .agg(
            mentions=("clean_text", "count"),
            newest=("created_at", "max"),
            oldest=("created_at", "min"),
            unique_authors=("author", "nunique")
        )
        .reset_index()
    )
    stats["days_span"] = (stats["newest"] - stats["oldest"]).dt.days
    return stats.sort_values("mentions", ascending=False)

def top_signal_phrases(df, theme, text_col="clean_text", top_k=8):
    g = df[df["market_theme"] == theme].copy()
    texts = g[text_col].astype(str).tolist()
    if len(texts) < 3:
        return []
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=8000)
    X = vec.fit_transform(texts)
    terms = np.array(vec.get_feature_names_out())
    mean_scores = np.asarray(X.mean(axis=0)).ravel()
    top_idx = np.argsort(mean_scores)[::-1][:top_k]
    return terms[top_idx].tolist()

def top_keywords_for_theme(df, theme, n=3):
    vals = df.loc[df["market_theme"] == theme, "keyword"].dropna().astype(str)
    if len(vals) == 0:
        return []
    return [k for k, _ in Counter(vals).most_common(n)]

In [84]:
def strip_urls(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", "", str(text))
    return re.sub(r"\s+", " ", text).strip()

def split_sentences(text: str):
    text = strip_urls(text)
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if 40 <= len(s.strip()) <= 240 and "hn:" not in s.lower()]

PAIN_WORDS = ["problem","pain","hard","difficult","issue","friction","annoy","struggle","fails","limitation","slow"]
OPP_WORDS  = ["opportunity","could","should","would","improve","better","build","need","demand","use case","workflow","automate","simplify","value"]
RISK_WORDS = ["risk","concern","legal","privacy","scam","fraud","ban","abuse","unsafe","security","trust","misuse","policy","compliance"]

def contains_any(sent, words):
    s = sent.lower()
    return any(w in s for w in words)

def build_actionable_insights(df, theme, n_each=3, max_rows=600):
    g = df[df["market_theme"] == theme].copy()
    if g.empty:
        return {"pain_signals": "", "opportunity_signals": "", "risk_signals": ""}

    g = g.sample(min(len(g), max_rows), random_state=42)

    rows = []
    for _, r in g.iterrows():
        for s in split_sentences(r.get("clean_text", "")):
            rows.append({"sent": s})
    sent_df = pd.DataFrame(rows).drop_duplicates(subset=["sent"])
    if sent_df.empty:
        return {"pain_signals": "", "opportunity_signals": "", "risk_signals": ""}

    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=6000)
    X = vec.fit_transform(sent_df["sent"].tolist())
    centroid = np.asarray(X.mean(axis=0))
    sims = cosine_similarity(X, centroid).ravel()
    sent_df["score"] = sims

    def pick_top(words):
        subset = sent_df[sent_df["sent"].apply(lambda s: contains_any(s, words))]
        subset = subset.sort_values("score", ascending=False)
        out = []
        for s in subset["sent"].tolist():
            if s.lower() not in [x.lower() for x in out]:
                out.append(s)
            if len(out) >= n_each:
                break
        return out

    def fallback():
        return sent_df.sort_values("score", ascending=False)["sent"].head(n_each).tolist()

    pain = pick_top(PAIN_WORDS) or fallback()
    opp  = pick_top(OPP_WORDS)  or fallback()
    risk = pick_top(RISK_WORDS) or fallback()

    return {
        "pain_signals": " | ".join(pain[:n_each]),
        "opportunity_signals": " | ".join(opp[:n_each]),
        "risk_signals": " | ".join(risk[:n_each]),
    }

In [85]:
stats = theme_stats(df)
rows = []

for _, r in stats.iterrows():
    theme = r["market_theme"]
    phrases = top_signal_phrases(df, theme, top_k=8)
    top_keys = top_keywords_for_theme(df, theme, n=3)

    q = quotes_df[quotes_df["market_theme"] == theme].sort_values("rank")
    sig = build_actionable_insights(df, theme)

    rows.append({
        "market_theme": theme,
        "mentions": int(r["mentions"]),
        "unique_authors": int(r["unique_authors"]) if pd.notna(r["unique_authors"]) else 0,
        "newest": r["newest"],
        "days_span": r["days_span"],
        "top_signal_phrases": ", ".join(phrases),
        "insight_summary": f"Conversation clusters around {', '.join(phrases[:4])}. Top source keywords: {', '.join(top_keys)}.",
        "quote_1": q.iloc[0]["clean_text"] if len(q) > 0 else "",
        "quote_1_title": q.iloc[0]["story_title"] if len(q) > 0 else "",
        "quote_1_url": q.iloc[0]["url"] if len(q) > 0 else "",
        "quote_2": q.iloc[1]["clean_text"] if len(q) > 1 else "",
        "quote_2_title": q.iloc[1]["story_title"] if len(q) > 1 else "",
        "quote_2_url": q.iloc[1]["url"] if len(q) > 1 else "",
        **sig
    })

insights_df = pd.DataFrame(rows).sort_values("mentions", ascending=False)

df.to_csv(OUT_SCORED, index=False)
insights_df.to_csv(OUT_INSIGHTS, index=False)
quotes_df.to_csv(OUT_QUOTES, index=False)

print("Saved:", OUT_SCORED, OUT_INSIGHTS, OUT_QUOTES)

Saved: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_scored_latest.csv /Users/adarshthakur/Downloads/MIANALYZER/data/insights_latest.csv /Users/adarshthakur/Downloads/MIANALYZER/data/quotes_latest.csv


In [86]:
df_scored = pd.read_csv(DATA_DIR / "hn_scored_latest.csv")

print(df_scored.shape)
df_scored[["market_theme"]].value_counts().head(10)

(874, 16)


market_theme                           
Early Product Feedback & Tool Discovery    381
Pricing & Monetization Concerns            203
Creator Economy & AI Disruption            137
Open Platform & Trust Signals               48
AI Content Platforms & Automation           46
Ad-Based Monetization Trade-offs            21
Developer Ecosystem & Integrations          20
Platform Perception & Social Sentiment      18
Name: count, dtype: int64

In [87]:
df_scored.sample(5)[
    ["market_theme", "keyword", "story_title", "clean_text"]
]

,market_theme,keyword,story_title,clean_text
145,Ad-Based Monetization Trade-offs,film,Nano Banana 2: Google's latest AI image genera...,Nano Banana 2: Google's latest AI image genera...
710,Pricing & Monetization Concerns,creator,Show HN: Vibevideo – Unified interface for top...,Show HN: Vibevideo – Unified interface for top...
18,Creator Economy & AI Disruption,film,LazyGravity – I made my phone control Antigrav...,LazyGravity – I made my phone control Antigrav...
499,Creator Economy & AI Disruption,platform,[dead],[dead]. I have just created a new AI social pl...
259,Creator Economy & AI Disruption,film,Show HN: Quoroom – local AI swarm (public rese...,Show HN: Quoroom – local AI swarm (public rese...


In [88]:
import pandas as pd

df = pd.read_csv(DATA_DIR /"hn_market_relevant_latest.csv")

df[["clean_text"]].head(10)

,clean_text
0,"Show HN: EloPhanto – Video creation, 116 tools..."
1,Show HN: I built an open-source analytics plat...
2,Show HN: OnGarde – Runtime content security pr...
3,Show HN: Portable Secret – Open-Source HTML fi...
4,Show HN: BetterDB Cloud – monitor Valkey/Redis...
5,Show HN: MCP server that checks if your projec...
6,Show HN: ArteSync – A package manager for AI c...
7,Show HN: Simple Viewers – Tiny native macOS fi...
8,Show HN: We built free adversarial security te...
9,Show HN: GalataJ – Runtime-aware Java profilin...


In [89]:
import pandas as pd
import re

df = pd.read_csv(DATA_DIR /"hn_scored_latest.csv")
df["clean_text"] = df["clean_text"].astype(str).fillna("")

def strip_urls(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

def split_sentences(text: str):
    text = strip_urls(text)
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if 40 <= len(s.strip()) <= 240]

def collect_theme_sentences(df, theme, text_col="clean_text"):
    g = df[df["market_theme"] == theme]
    sentences = []
    for t in g[text_col].astype(str):
        sentences.extend(split_sentences(t))
    # dedupe while preserving order
    return list(dict.fromkeys(sentences))

SIGNAL_WORDS = [
    "problem","pain","hard","difficult","issue","frustrat",
    "need","gap","missing","lack",
    "opportunity","should","could","better","build","improve",
    "monetization","pricing","workflow","platform","creator"
]

def signal_score(sentence):
    s = sentence.lower()
    return sum(1 for w in SIGNAL_WORDS if w in s)

# build the same "final_sentences" idea
all_sentences = []
for theme in df["market_theme"].dropna().unique():
    sents = collect_theme_sentences(df, theme)
    for s in sents:
        sc = signal_score(s)
        if sc > 0:
            all_sentences.append({"sentence": s, "theme": theme, "score": sc})

sent_df = pd.DataFrame(all_sentences).sort_values("score", ascending=False)

final_sentences = []
used_themes = set()
for _, row in sent_df.iterrows():
    if len(final_sentences) == 5:
        break
    if row["theme"] not in used_themes:
        final_sentences.append(row["sentence"])
        used_themes.add(row["theme"])

for i, s in enumerate(final_sentences, start=1):
    print(f"{i}. {s}")

1. Hi HN, I build data platforms (Snowflake, dbt, Airflow) and kept seeing the same issue: starting a clean analytics stack is harder than it should be.
2. Over the past few months, I've been working on building Omni - a workplace search and chat platform that connects to apps like Google Drive/Gmail, Slack, Confluence, etc.
3. If we want parents to be accountable, then these platforms need to provide better tools to enable parents to do so.
4. I have it in my head that a lot of these problems core issue is a lack of faith / effort in creating good front line management.
5. I’d say the mandatory software platform I need for my bank, drivers license, daily communication, etc should be in this bucket.


In [90]:
import pandas as pd
import re

df = pd.read_csv(DATA_DIR /"hn_scored_latest.csv")
df["clean_text"] = df["clean_text"].astype(str).fillna("")

def strip_urls(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

def split_sentences(text: str):
    text = strip_urls(text)
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if 40 <= len(s.strip()) <= 260]

PAIN_CUES = [
    "problem", "issue", "pain", "hard", "difficult", "friction",
    "annoy", "struggle", "fails", "broken", "missing", "lack",
    "can't", "cannot", "won't", "requires", "need to", "have to",
    "too expensive", "expensive", "time consuming", "manual", "blocked",
    "privacy", "trust", "scam", "fraud"
]

DOMAIN_CUES = [
    "platform", "workflow", "automation", "tool", "api", "integration",
    "monetization", "pricing", "subscription", "ads", "creator", "creators",
    "users", "product", "open source", "closed", "permissions", "access",
    "google", "oauth", "privacy", "trust"
]

def score_sentence(s: str) -> int:
    sl = s.lower()
    return sum(w in sl for w in PAIN_CUES) + sum(w in sl for w in DOMAIN_CUES)

# Build a sentence table
rows = []
for _, r in df.iterrows():
    theme = r.get("market_theme", "")
    for s in split_sentences(r.get("clean_text", "")):
        sl = s.lower()
        if any(w in sl for w in PAIN_CUES) and any(w in sl for w in DOMAIN_CUES):
            rows.append({"theme": theme, "sentence": s, "score": score_sentence(s)})

sent_df = pd.DataFrame(rows).drop_duplicates(subset=["sentence"])
sent_df = sent_df.sort_values("score", ascending=False)

# Pick 5 sentences from 5 different themes (like before)
final = []
used = set()
for _, row in sent_df.iterrows():
    if len(final) == 5:
        break
    if row["theme"] not in used:
        final.append((row["theme"], row["sentence"]))
        used.add(row["theme"])

for i, (theme, s) in enumerate(final, start=1):
    print(f"{i}. [{theme}] {s}")

1. [Pricing & Monetization Concerns] I built this because many citation tools return incomplete APA references (especially missing author/date -> n.d.), and users still have to fix everything manually.
2. [AI Content Platforms & Automation] The privacy issues of data retention on Google/Meta/etc social and SaaS platforms is something to care about but it is only a small piece of the puzzle of data privacy.
3. [Creator Economy & AI Disruption] Beads[1] (Steve Yegge's git-native issue tracking for agents) has been a great boost to my agents' productivity, but it's also made them more difficult to keep aligned.
4. [Open Platform & Trust Signals] >Some observers present privacy-preserving age proofs involving a third party, such as the government, as a solution, but they inherit the same structural flaw: many users who are legally old enough to use a platform do not have government ID.
5. [Early Product Feedback & Tool Discovery] Unfortunately things are going in the opposite direction wit